### Importação de bibliotecas 

In [1]:

from IPython import get_ipython

import pandas as pd 
import os
import dotenv
from pathlib import Path
from pandas import DataFrame

dotenv.load_dotenv()


try:
    from src.data.utils import gerar_df_dic,carregar_parquet_local
except ModuleNotFoundError:
    # Se falhar, instala o pacote local e importa
    get_ipython().run_line_magic('pip', 'install -e ..')
    from src.data.utils import gerar_df_dic,carregar_parquet_local

### Leitura de arquivos (CSV)

In [2]:
ano = 2023

#### TS_ALUNO <br>
> Informações dos alunos que participaram das avaliações estaduais do 2º ano do ensino fundamental.

In [3]:
raw_alunos, dicionario_alunos = gerar_df_dic(ano,'TS_ALUNO')
raw_alunos.head(5)
dicionario_alunos


Lendo dicionário em: ..\data\raw\dados_2023\DICIONÁRIO\Dicionario_Microdados_AEEB_2023.xlsx


,Descrição,Variáveis Categóricas,Tamanho,Tipo
Variável,,,,
NU_ANO_AVALIACAO,Ano de aplicação da avaliação estadual,nan,4,Numérica
CO_UF,Código da Unidade da Federação,nan,2,Numérica
SG_UF,Sigla da Unidade da Federação,nan,2,Alfanumérica
ID_ALUNO,Identificador do Aluno,nan,8,Numérica
TP_SERIE,Ano Escolar,2 - 2º ano do Ensino Fundamental,1,Numérica
ID_ESCOLA,Máscara do Códigoo da Escola (códigos fictícios),nan,8,Numérica
TP_DEPENDENCIA,Dependência Administrativa da Escola,1 - Federal;\n2 - Estadual;\n3 - Municipal;\n4...,1,Numérica
CO_MUNICIPIO,Código do Município onde está localizada a esc...,nan,7,Numérica
NO_MUNICIPIO,Nome do Município onde está localizada a escola.,nan,150,Alfanumérica


#### TS_ITEM <br>
> Informações dos itens de provas utilizados na avaliação estadual do 2º ano do ensino fundamental para a disciplina Língua Portuguesa (LP).

In [4]:
raw_itens, dicionario_itens = gerar_df_dic(ano,'TS_ITEM')
raw_itens.head(5)
dicionario_itens.head(5)

Lendo dicionário em: ..\data\raw\dados_2023\DICIONÁRIO\Dicionario_Microdados_AEEB_2023.xlsx


,Descrição,Variáveis Categóricas,Tamanho,Tipo
Variável,,,,
NU_ANO_AVALIACAO,Ano de aplicação da avaliação estadual,nan,4,Numérica
CO_UF,Código da Unidade da Federação,nan,2,Numérica
SG_UF,Sigla da Unidade da Federação,nan,2,Alfanumérica
CO_BLOCO,Código do Bloco,nan,2,Numérica
NU_POSICAO,Posição do item no Bloco.,nan,2,Numérica


#### TS_ESTADO

> Resultados das avaliações estaduais por UF do 2º ano do ensino fundamental para a disciplina Língua Portuguesa (LP), equalizadas com o SAEB.

In [8]:
#raw_uf, dicionario_uf = gerar_df_dic(ano,'TS_ESTADO')
# raw_uf.head(5)
# dicionario_uf.head(5)


Lendo dicionário em: ..\data\raw\dados_2023\DICIONÁRIO\Dicionario_Microdados_AEEB_2023.xlsx


#### TS_MUNICIPIO 

> Resultados das avaliações estaduais por município do 2º ano do ensino fundamental para a disciplina Língua Portuguesa (LP), equalizadas com o SAEB.

In [9]:
#raw_municipio, dicionario_municipio = gerar_df_dic(2023,'TS_MUNICIPIO')
# raw_municipio.head(5)
# dicionario_municipio.head(5)

Lendo dicionário em: ..\data\raw\dados_2023\DICIONÁRIO\Dicionario_Microdados_AEEB_2023.xlsx


### Analise da documentação do Inep (Leiame)

**raw_uf** <br>
*TS_ESTADO: Apresenta os resultados das avaliações estaduais do 2º ano do ensino fundamental para Língua Portuguesa (LP), também equalizados com o Saeb, mas neste arquivo os dados estão agregados por Estado (Unidade da Federação)*

**raw_municipio** <br>
*TS_MUNICIPIO : Contém os resultados das avaliações estaduais do 2º ano do ensino fundamental para a disciplina de Língua Portuguesa (LP) agregados em nível municipal
. Estes resultados são equalizados com os do Sistema de Avaliação da Educação Básica (Saeb)*

**raw_itens** <br>
*TS_ITEM : Contém o cadastro e as informações dos itens (questões) das provas utilizados na avaliação estadual do 2º ano para a disciplina de Língua Portuguesa*

**raw_alunos** <br> 
*TS_ALUNO:   é uma tabela de microdados que contém as informações dos alunos que participaram das avaliações estaduais do 2º ano do ensino fundamental os dados dos alunos seguem as leis de LGPD (Não há dados pessoais).*





**Breve descrição do que foi encontrado nos documentos técnicos LEIAME da base de dados**

Houve uma diferença no tratamento de qualidade de exclusão e inconsistência ao longo dos anos 
faremos uma breve analise de como isso foi tratado antes de nossa analise pelo Saeb: 

**2023**

- **TS_ALUNO** ->não existem dados de São Paulo na tabela <br><br>
- **TS_ALUNO** -> 4 registros foram retirados em razão de teremo 743 na proeficiencia (VL_PROFICIENCIA_LP) e 0 na maracação de alfabetização  <br><br>
- **TS_ITEM** -> A base foi gerada sem as informações de Mato Grosso do Sul, São Paulo e Sergipe 

**2024** 

- **TS_ALUNO** ->  Houve a exclusão de 224 registros de alunos de sete escolas (no Amazonas, Espírito Santo e São Paulo) que não foram localizadas no Censo Escolar 2024
. Foram excluídos também 67 registros devido à mesma inconsistência do ano anterior (proficiência 743, mas 0 em alfabetizado)
. Iniciado neste ano foi a dos códigos reais das escolas por máscaras, para seguir o padrão de proteção LGPD <br><br>
- **TS_MUNICIPIO** -> Foram retirados 3 registros onde a variável de percentual de aluno alfabetizado (PC_ALUNO_ALFABETIZADO) era igual a zero  <br><br>
- **TS_ITEM** ->  Itens da Bahia, São Paulo e Rio Grande do Sul foram retirados porque seus parâmetros não pareciam estar na escala do Saeb
. A Bahia também não enviou os itens comuns<br><br>
- **TS_ESTADO** -> Roraima (que não realizou a avaliação em 2024) e o Distrito Federal não enviaram dados
. Os resultados da rede estadual do Distrito Federal ficaram associados apenas à base TS_MUNICIPIO

**2025**

- **TS_ALUNO** ->Foram retirados 628 registros que estavam com a informação da escola em branco (Missing), ligados a dez escolas (em Rondônia, Pernambuco, Espírito Santo, São Paulo e Mato Grosso) que não foram achadas no Censo Escolar 2025
. A política de mascarar os códigos das escolas continuou a ser aplicada<br><br>

- **TS_MUNICIPIO** ->Foram retirados 7 registros com a variável PC_ALUNO_ALFABETIZADO zerada. <br><br>

- **TS_ITEM** -> Assim como no ano anterior, itens de São Paulo e do Rio Grande do Sul foram retirados por não estarem na escala correta do Saeb


### EDA

Para simplificar iremos carregar todos os dados a partir daqui já na camada bronze (.parquet) 
podemos ignorar a primeira leitura de arquivos a partir daqui caso já tenha sido rodado o pipeline_s3, irei fazer as analises todas locais num primeiro momento, depois irei integrar com a nuvem

In [7]:
#Nao ocultar as informações de descrição 
pd.set_option('display.max_colwidth', None)

ano = 2025

#dados
raw_alunos = carregar_parquet_local(ano,'TS_ALUNO')
raw_itens =  carregar_parquet_local(ano,'TS_ITEM') 
raw_uf  = carregar_parquet_local(ano,'TS_ESTADO') 
raw_municipio  = carregar_parquet_local(ano,'TS_MUNICIPIO') 


#Dicionarios
dicionario_alunos =   carregar_parquet_local(ano,'TS_ALUNO', ler_dicionario=True)
dicionario_itens =   carregar_parquet_local(ano,'TS_ITEM', ler_dicionario=True)
dicionario_municipio =   carregar_parquet_local(ano,'TS_ESTADO', ler_dicionario=True)
dicionario_uf =   carregar_parquet_local(ano,'TS_MUNICIPIO', ler_dicionario=True)

Lendo Parquet em: ..\data\bronze\ano=2025\dados\TS_ALUNO.parquet
Lendo Parquet em: ..\data\bronze\ano=2025\dados\TS_ITEM.parquet
Lendo Parquet em: ..\data\bronze\ano=2025\dados\TS_ESTADO.parquet
Lendo Parquet em: ..\data\bronze\ano=2025\dados\TS_MUNICIPIO.parquet
Lendo Parquet em: ..\data\bronze\ano=2025\dicionario\dicionario_TS_ALUNO.parquet
Lendo Parquet em: ..\data\bronze\ano=2025\dicionario\dicionario_TS_ITEM.parquet
Lendo Parquet em: ..\data\bronze\ano=2025\dicionario\dicionario_TS_ESTADO.parquet
Lendo Parquet em: ..\data\bronze\ano=2025\dicionario\dicionario_TS_MUNICIPIO.parquet


feita uma verificação dos valores nulos junto aos dicionarios para entender o que eles significam 

In [152]:
raw_municipio.isna().sum().sum()

0

In [153]:
raw_uf.isna().sum().sum()

0

In [154]:
itens_na = pd.DataFrame(raw_itens.isna().sum(), columns=['Qtd_Nulos'])

itens_na['Descrição'] = itens_na.index.map(dicionario_itens['Descrição'])

itens_na_filtrado = itens_na[itens_na['Qtd_Nulos'] > 0]
itens_na_filtrado



,Qtd_Nulos,Descrição
DS_GABARITO,660,Gabarito do Item
NU_PARAM_A,6,Parâmetro de discriminação: é o poder de discriminação do item para diferenciar os participantes que dominam dos participantes que não dominam a habilidade avaliada.
NU_PARAM_B,666,"Parâmetro de dificuldade: associado à dificuldade do item, sendo que quanto maior seu valor, mais difícil é o item."
NU_PARAM_C,666,Parâmetro de acerto ao acaso: é a probabilidade de um participante acertar o item não dominando a habilidade exigida.
NU_PARAM_B1,1300,Parâmetro de dificuldade da transição entre as categorias de “erro” e “acerto parcial”.
NU_PARAM_B2,1304,Parâmetro de dificuldade da transição entre as categorias de “acerto parcial” e “acerto total”.
NU_PARAM_B3,1594,Parâmetro de dificuldade da transição entre as categorias de “acerto parcial” e “acerto total”.
NU_PARAM_B4,1810,Parâmetro de dificuldade da transição entre as categorias de “acerto parcial” e “acerto total”.


Os itens possuem algumas variaveis nulas, aparentemente por conta de calculos, como niveis de dificuldade ou probabilidade de acerto em caso de chuta, assim são que não são possiveis de calcular ou ainda questões canceladas 

In [155]:
alunos_na = pd.DataFrame(raw_alunos.isna().sum(), columns=['Qtd_Nulos'])

alunos_na['Descrição'] = alunos_na.index.map(dicionario_alunos['Descrição'])

alunos_na_filtrado = alunos_na[alunos_na['Qtd_Nulos'] > 0]
alunos_na_filtrado


,Qtd_Nulos,Descrição
ID_ESCOLA,628,Máscara do Códigoo da Escola (códigos fictícios)
TP_DEPENDENCIA,628,Dependência Administrativa da Escola
CO_MUNICIPIO,628,Código do Município onde está localizada a escola.
NO_MUNICIPIO,628,Nome do Município onde está localizada a escola.
CO_BLOCO_1,655732,"Código do Bloco na posição 1 do Caderno atribuído ao aluno na prova de Língua Portuguesa (LP). Observação: Apenas se o bloco contiver somente itens de resposta objetiva. Caso contrário, a informação ficará em branco (Missing)."
TX_RESPOSTA_BLOCO_1,659048,Vetor com as respostas dos itens objetivos do bloco na posição 1 do Caderno atribuído ao aluno na prova de Língua Portuguesa (LP).
TX_GABARITO_BLOCO_1,655732,Vetor com o gabarito dos itens objetivos do bloco na posição 1 do Caderno atribuído ao aluno na prova de Língua Portuguesa (LP).
CO_BLOCO_2,252871,NaN
TX_RESPOSTA_BLOCO_2,256187,Vetor com as respostas dos itens objetivos do bloco na posição 2 do Caderno atribuído ao aluno na prova de Língua Portuguesa (LP).
TX_GABARITO_BLOCO_2,252871,Vetor com o gabarito dos itens objetivos do bloco na posição 2 do Caderno atribuído ao aluno na prova de Língua Portuguesa (LP).


**Valores Nulos encontrados nas tabelas Alunos**

VL_PESO_ALUNO_LP :		Peso do aluno na prova de Língua Portuguesa <br>
VL_PROFICIENCIA_LP :		Proficiência do aluno em Língua Portuguesa (LP) calculada na escala equalizada com o SAEB.

costumam ser identificos e tratam as notas do aluno, dessa forma entende-se que os valores nulos se dão pela ausencia do aluno na prova, confirmados pela variavel IN_PRESENCA_LP como 0 = ausente

a partir de 2025 contamos também com blocos para cada prova, por isso se da um maior numero de nulos, na pratica o aluno com Caderno 1 responde o bloco 1 e 2, mas nao responde o 3 e 4, sendo assim esses são nulos.

In [156]:
total_alunos = raw_alunos.shape[0]
abstencao = (raw_alunos['IN_PRESENCA_LP'] == 0).sum()
total_sem_nota = raw_alunos['VL_PROFICIENCIA_LP'].isna().sum()
presentes_sem_nota = total_sem_nota - abstencao

print(f"Total de alunos na base : {total_alunos}")
print(f"Abstenção (Faltaram)    : {abstencao}")
print(f"Total sem nota final    : {total_sem_nota}")
print(f"Presentes sem nota      : {presentes_sem_nota}")


Total de alunos na base : 2222792
Abstenção (Faltaram)    : 252871
Total sem nota final    : 256187
Presentes sem nota      : 3316
